# Introduction to PySpark

3 bộ dữ liệu đi kèm:

- `adults.json`
- `salaries.csv`
- `Monthly_Transportation_Statistics.csv`

## Mục tiêu
Sinh viên có thể:
1. Khởi tạo `SparkSession`.
2. Đọc CSV/JSON và kiểm tra schema.
3. Dùng `select`, `filter`, `where`, `sort`, `groupBy`, `agg`.
4. Xử lý missing data và thao tác cột.
5. Thực hiện `join`, `union`.
6. Viết UDF cơ bản.
7. Hiểu RDD và các transformation/action.
8. Dùng Spark SQL.
9. Dùng cache/persist, broadcast và `explain()`.

## 0. Cài đặt và kiểm tra môi trường

In [ ]:
# Nếu máy chưa có PySpark, bỏ dấu # ở dòng dưới:
# %pip install -q pyspark

import sys
print("Python:", sys.version)

## 1. Khởi tạo SparkSession

`SparkSession` là điểm vào chính để làm việc với DataFrame và Spark SQL.

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("IntroductionToPySpark")
    .master("local[*]")
    .getOrCreate()
)

print("Spark version:", spark.version)
print("Spark master :", spark.sparkContext.master)

- `builder`: cấu hình phiên Spark.
- `appName(...)`: đặt tên ứng dụng.
- `master("local[*]")`: chạy local trên các CPU core.
- `getOrCreate()`: tạo mới hoặc lấy session hiện có.

## 2. Đọc JSON — `adults.json`

In [ ]:
from pathlib import Path
from pyspark.sql import functions as F

DATA_DIR = Path(".")
ADULTS_FILE = str(DATA_DIR / "adults.json")

adults_df = spark.read.json(ADULTS_FILE)
adults_df.show(5, truncate=False)

### 2.1 Kiểm tra schema

In [ ]:
adults_df.printSchema()

Tên cột có dấu chấm như `education.num` cần dùng backtick khi tham chiếu bằng `col()`.

In [ ]:
adults_df.select(
    "age",
    F.col("`education.num`").alias("education_num"),
    "occupation",
    "income"
).show(5)

## 3. DataFrame cơ bản: select, filter, where, sort

In [ ]:
adults_df.select("age", "occupation", "income").show(10)

In [ ]:
adults_over_50 = (
    adults_df
    .filter(F.col("age") > 50)
    .select("age", "occupation", "income")
)
adults_over_50.show(10)

In [ ]:
adults_df.where(F.col("income") == ">50K").show(10)

In [ ]:
adults_df.orderBy(F.col("age").desc()).show(10)

### Bài tập 1
1. Lọc `age >= 40`.
2. Chỉ giữ `age`, `occupation`, `income`.
3. Sắp xếp tuổi giảm dần.
4. Hiển thị 15 dòng.

In [ ]:
# TODO - Bài tập 1

## 4. Đọc CSV — `salaries.csv`

In [ ]:
SALARIES_FILE = str(DATA_DIR / "salaries.csv")

salaries_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(SALARIES_FILE)
)

salaries_df.show(5, truncate=False)

In [ ]:
salaries_df.printSchema()
print("Số dòng:", salaries_df.count())
print("Số cột :", len(salaries_df.columns))

### 4.1 Select và filter

In [ ]:
salaries_df.select(
    "work_year", "experience_level", "job_title",
    "salary_in_usd", "remote_ratio"
).show(10, truncate=False)

In [ ]:
high_salary_df = (
    salaries_df
    .filter(F.col("salary_in_usd") >= 200000)
    .select("work_year", "experience_level", "job_title", "salary_in_usd")
    .orderBy(F.col("salary_in_usd").desc())
)
high_salary_df.show(20, truncate=False)

## 5. Aggregation: groupBy và agg

In [ ]:
from pyspark.sql.functions import avg, min, max, count, round as spark_round

salary_by_experience = (
    salaries_df
    .groupBy("experience_level")
    .agg(
        count("*").alias("n"),
        spark_round(avg("salary_in_usd"), 2).alias("avg_salary_usd"),
        min("salary_in_usd").alias("min_salary_usd"),
        max("salary_in_usd").alias("max_salary_usd")
    )
    .orderBy("experience_level")
)
salary_by_experience.show()

In [ ]:
(
    salaries_df
    .groupBy("work_year")
    .agg(
        count("*").alias("n"),
        spark_round(avg("salary_in_usd"), 2).alias("avg_salary_usd")
    )
    .orderBy("work_year")
    .show()
)

### Bài tập 2
Tính theo `company_size`: số bản ghi, lương trung bình, lương lớn nhất.
Sắp xếp theo lương trung bình giảm dần.

In [ ]:
# TODO - Bài tập 2

## 6. Missing data

In [ ]:
null_counts = salaries_df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in salaries_df.columns
])
null_counts.show(truncate=False)

In [ ]:
salaries_dropna = salaries_df.na.drop()
print("Trước:", salaries_df.count())
print("Sau :", salaries_dropna.count())

Trong `adults.json`, một số giá trị thiếu của `occupation` được ký hiệu bằng `"?"`.
Chuẩn hóa `"?"` thành `null` trước khi xử lý.

In [ ]:
adults_clean = adults_df.withColumn(
    "occupation",
    F.when(F.col("occupation") == "?", None).otherwise(F.col("occupation"))
)

adults_clean.filter(F.col("occupation").isNull()).show(10)

In [ ]:
adults_filled = adults_clean.na.fill({"occupation": "Unknown"})
adults_filled.show(10)

## 7. Thao tác cột: withColumn, rename, drop

In [ ]:
salary_features = (
    salaries_df
    .withColumn("salary_k_usd", F.round(F.col("salary_in_usd") / 1000, 2))
    .withColumn(
        "remote_type",
        F.when(F.col("remote_ratio") == 100, "Remote")
         .when(F.col("remote_ratio") == 50, "Hybrid")
         .otherwise("On-site")
    )
)

salary_features.select(
    "job_title", "salary_in_usd", "salary_k_usd",
    "remote_ratio", "remote_type"
).show(10, truncate=False)

In [ ]:
renamed_df = salary_features.withColumnRenamed("salary_in_usd", "salary_usd")
renamed_df.select("job_title", "salary_usd").show(5, truncate=False)

In [ ]:
dropped_df = salary_features.drop("salary", "salary_currency")
print(dropped_df.columns)

## 8. Join

In [ ]:
experience_lookup = spark.createDataFrame(
    [
        ("EN", "Entry-level / Junior"),
        ("MI", "Mid-level / Intermediate"),
        ("SE", "Senior / Expert"),
        ("EX", "Executive-level / Director"),
    ],
    ["experience_level", "experience_label"]
)

experience_lookup.show()

In [ ]:
salary_joined = salaries_df.join(
    experience_lookup,
    on="experience_level",
    how="left"
)

salary_joined.select(
    "experience_level", "experience_label",
    "job_title", "salary_in_usd"
).show(10, truncate=False)

## 9. Union

In [ ]:
sample_a = salaries_df.filter(F.col("work_year") == 2020).limit(5)
sample_b = salaries_df.filter(F.col("work_year") == 2021).limit(5)

sample_union = sample_a.union(sample_b)
print("Số dòng sau union:", sample_union.count())

sample_union.select("work_year", "job_title", "salary_in_usd").show(truncate=False)

## 10. UDF — User Defined Function

Trong thực tế, nên ưu tiên các hàm có sẵn trong `pyspark.sql.functions`
trước khi dùng Python UDF.

In [ ]:
from pyspark.sql.types import StringType
from pyspark.sql.functions import udf

def salary_band(salary):
    if salary is None:
        return None
    if salary < 50000:
        return "Low"
    elif salary < 100000:
        return "Medium"
    elif salary < 200000:
        return "High"
    return "Very High"

salary_band_udf = udf(salary_band, StringType())

salary_udf_df = salaries_df.withColumn(
    "salary_band",
    salary_band_udf(F.col("salary_in_usd"))
)

salary_udf_df.select(
    "job_title", "salary_in_usd", "salary_band"
).show(10, truncate=False)

### 10.1 Cách ưu tiên hơn: Spark native expression

In [ ]:
salary_native_df = salaries_df.withColumn(
    "salary_band",
    F.when(F.col("salary_in_usd") < 50000, "Low")
     .when(F.col("salary_in_usd") < 100000, "Medium")
     .when(F.col("salary_in_usd") < 200000, "High")
     .otherwise("Very High")
)

salary_native_df.select(
    "job_title", "salary_in_usd", "salary_band"
).show(10, truncate=False)

## 11. RDD — Resilient Distributed Dataset

- Transformation: `map`, `filter`, `flatMap`, `reduceByKey`, ...
- Action: `collect`, `count`, `take`, `reduce`, ...

> Không nên `collect()` toàn bộ dữ liệu lớn về driver.

In [ ]:
adults_rdd = adults_df.rdd

print("Kiểu:", type(adults_rdd))
print("Số phần tử:", adults_rdd.count())

for row in adults_rdd.take(3):
    print(row)

In [ ]:
ages_rdd = adults_rdd.map(lambda row: row["age"])
print("5 tuổi đầu:", ages_rdd.take(5))

In [ ]:
older_rdd = adults_rdd.filter(lambda row: row["age"] >= 60)
print("Số người >= 60:", older_rdd.count())
print(older_rdd.take(5))

In [ ]:
income_count_rdd = (
    adults_rdd
    .map(lambda row: (row["income"], 1))
    .reduceByKey(lambda x, y: x + y)
)

print(income_count_rdd.collect())

## 12. DataFrame vs RDD

| DataFrame | RDD |
|---|---|
| API mức cao | API mức thấp |
| Có schema | Không có schema dạng bảng rõ ràng |
| Hỗ trợ Spark SQL | Không trực tiếp |
| Có optimizer | Ít tối ưu tự động hơn |
| Phù hợp phần lớn ETL/analytics | Dùng khi cần xử lý mức thấp |

**Khuyến nghị:** ưu tiên DataFrame cho phần lớn bài toán phân tích dữ liệu.

## 13. Spark SQL

In [ ]:
salaries_df.createOrReplaceTempView("salaries")

spark.sql("""
SELECT
    experience_level,
    COUNT(*) AS n,
    ROUND(AVG(salary_in_usd), 2) AS avg_salary_usd
FROM salaries
GROUP BY experience_level
ORDER BY avg_salary_usd DESC
""").show()

### 13.1 Kết hợp SQL và DataFrame API

In [ ]:
sql_result = spark.sql("""
SELECT *
FROM salaries
WHERE salary_in_usd >= 150000
""")

(
    sql_result
    .withColumn("salary_k_usd", F.round(F.col("salary_in_usd") / 1000, 1))
    .select("job_title", "experience_level", "salary_in_usd", "salary_k_usd")
    .orderBy(F.col("salary_in_usd").desc())
    .show(15, truncate=False)
)

### Bài tập 3 — Spark SQL
Tìm 10 `job_title` có:
- ít nhất 20 bản ghi,
- lương trung bình cao nhất.

Gợi ý: `GROUP BY job_title`, `HAVING COUNT(*) >= 20`.

In [ ]:
# TODO - Bài tập 3

## 14. Dữ liệu giao thông nhiều cột

Dataset này phù hợp để luyện chọn cột, missing data, parsing thời gian,
cache/persist và execution plan.

In [ ]:
TRANSPORT_FILE = str(DATA_DIR / "Monthly_Transportation_Statistics.csv")

transport_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(TRANSPORT_FILE)
)

print("Số dòng:", transport_df.count())
print("Số cột :", len(transport_df.columns))
transport_df.select("Index", "Date").show(5, truncate=False)

In [ ]:
transport_small = transport_df.select(
    "Index",
    "Date",
    "Highway Fatalities",
    "Highway Fuel Price - Regular Gasoline",
    "Unemployment Rate - Seasonally Adjusted",
    "U.S. Airline Traffic - Total - Seasonally Adjusted"
)

transport_small.show(10, truncate=False)

In [ ]:
transport_nulls = transport_small.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in transport_small.columns
])

transport_nulls.show(truncate=False)

In [ ]:
transport_typed = transport_small.withColumn(
    "DateParsed",
    F.to_timestamp("Date", "MM/dd/yyyy hh:mm:ss a")
)

transport_typed.select("Date", "DateParsed").show(5, truncate=False)

## 15. Execution plan và lazy evaluation

Spark dùng lazy evaluation: transformation chưa nhất thiết chạy ngay.
Action như `show()`, `count()`, `collect()`, `write...` mới kích hoạt job.

In [ ]:
plan_df = (
    salaries_df
    .filter(F.col("salary_in_usd") > 100000)
    .select("experience_level", "job_title", "salary_in_usd")
)

plan_df.explain(mode="formatted")

## 16. Cache và Persist

In [ ]:
cached_salary = salaries_df.cache()

print("Rows:", cached_salary.count())
cached_salary.groupBy("experience_level").avg("salary_in_usd").show()
cached_salary.groupBy("company_size").count().show()

cached_salary.unpersist()

In [ ]:
from pyspark import StorageLevel

transport_cached = transport_df.persist(StorageLevel.MEMORY_AND_DISK)

print("Rows:", transport_cached.count())
transport_cached.select("Date", "Highway Fatalities").show(5)

transport_cached.unpersist()

## 17. Broadcast Join

In [ ]:
from pyspark.sql.functions import broadcast

broadcast_joined = salaries_df.join(
    broadcast(experience_lookup),
    on="experience_level",
    how="left"
)

broadcast_joined.select(
    "experience_level", "experience_label", "job_title"
).show(10, truncate=False)

## 18. Mini Lab — Phân tích dữ liệu lương bằng PySpark

1. Dataset có bao nhiêu dòng?
2. Có bao nhiêu `job_title` khác nhau?
3. Top 10 `job_title` có nhiều bản ghi nhất.
4. Lương trung bình theo `experience_level`.
5. Lương trung bình theo `remote_ratio`.
6. Top 10 `job_title` có lương trung bình cao nhất, chỉ xét chức danh có ít nhất 30 bản ghi.
7. Làm câu 6 bằng DataFrame API và Spark SQL.

In [ ]:
print("Số dòng:", salaries_df.count())
print(
    "Số job title khác nhau:",
    salaries_df.select("job_title").distinct().count()
)

In [ ]:
(
    salaries_df
    .groupBy("job_title")
    .count()
    .orderBy(F.col("count").desc())
    .show(10, truncate=False)
)

In [ ]:
(
    salaries_df
    .groupBy("experience_level")
    .agg(F.round(F.avg("salary_in_usd"), 2).alias("avg_salary_usd"))
    .orderBy(F.col("avg_salary_usd").desc())
    .show()
)

In [ ]:
(
    salaries_df
    .groupBy("remote_ratio")
    .agg(
        F.count("*").alias("n"),
        F.round(F.avg("salary_in_usd"), 2).alias("avg_salary_usd")
    )
    .orderBy("remote_ratio")
    .show()
)

In [ ]:
(
    salaries_df
    .groupBy("job_title")
    .agg(
        F.count("*").alias("n"),
        F.round(F.avg("salary_in_usd"), 2).alias("avg_salary_usd")
    )
    .filter(F.col("n") >= 30)
    .orderBy(F.col("avg_salary_usd").desc())
    .show(10, truncate=False)
)

In [ ]:
spark.sql("""
SELECT
    job_title,
    COUNT(*) AS n,
    ROUND(AVG(salary_in_usd), 2) AS avg_salary_usd
FROM salaries
GROUP BY job_title
HAVING COUNT(*) >= 30
ORDER BY avg_salary_usd DESC
LIMIT 10
""").show(truncate=False)

## 19. Bài tập tự luyện

### A. Adults
1. Đếm theo `income`.
2. Tuổi trung bình theo `income`.
3. Đếm theo `marital.status`.
4. Chuẩn hóa `occupation = "?"` thành null.
5. Tìm 5 occupation phổ biến nhất.

### B. Salaries
1. Top 10 chức danh có lương trung bình cao nhất.
2. Lương trung bình theo `company_size`.
3. Lương trung bình theo `work_year` và `experience_level`.
4. Tạo cột phân nhóm mức lương.
5. So sánh Remote/Hybrid/On-site.

### C. Transportation
1. Chọn 8–10 cột có ý nghĩa.
2. Đếm missing value.
3. Chỉ giữ dữ liệu từ năm 2000.
4. Tính trung bình theo năm cho một chỉ số.
5. Giải thích execution plan.

## 20. Cheatsheet

```python
# Đọc dữ liệu
spark.read.csv(...)
spark.read.json(...)
spark.read.parquet(...)

# Khám phá
df.show()
df.printSchema()
df.count()

# DataFrame
df.select(...)
df.filter(...)
df.where(...)
df.orderBy(...)

# Missing
df.na.drop()
df.na.fill({...})
df.filter(F.col("x").isNotNull())

# Cột
df.withColumn(...)
df.withColumnRenamed(...)
df.drop(...)

# Tổng hợp
df.groupBy(...).agg(...)

# Kết hợp
df1.join(df2, ..., how="inner")
df1.union(df2)

# SQL
df.createOrReplaceTempView("table")
spark.sql("SELECT ...")

# RDD
df.rdd
rdd.map(...)
rdd.filter(...)
rdd.reduceByKey(...)
rdd.take(...)

# Hiệu năng
df.explain()
df.cache()
df.persist(...)
df.unpersist()
broadcast(df)
```

## 21. Kết thúc SparkSession

In [ ]:
# spark.stop()